In [1]:
import anthropic
import instructor
from qdrant_client import QdrantClient
from pydantic import BaseModel, Field
import voyageai
from dotenv import load_dotenv
import os

/Users/pranjal/Desktop/projects/AI/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### RAG Pipeline

In [7]:
class RAGGenerationResponse(BaseModel):
        answer: str = Field(description="The answer to the question")

In [2]:
instructor_client = instructor.from_anthropic(anthropic.Anthropic())
anthropic_client = anthropic.Anthropic()
qdrant_client = QdrantClient(url='http://localhost:6333')

In [12]:
def get_embedding(voyageai_client, text, model = 'voyage-3'):
        result = voyageai_client.embed(
                [text],
                model=model,
                input_type="document"
        )
        return result.embeddings[0]


def retrieve_data(voyageai_client, query, qdrant_client, k=5):
        query_embedding = get_embedding(voyageai_client, query)
        results = qdrant_client.query_points(
                collection_name="Amazon-items-collection-00",
                query=query_embedding,
                limit=k,
        )

        retrieved_context_ids = []
        retrieved_context = []
        similarity_scores = []
        retrieved_context_ratings = []

        for item in results.points:
                retrieved_context_ids.append(item.payload['parent_asin'])
                retrieved_context.append(item.payload['description'])
                retrieved_context_ratings.append(item.payload['average_rating'])
                similarity_scores.append(item.score)
        
        return {
                "retrieved_context_ids": retrieved_context_ids,
                "retrieved_context": retrieved_context,
                "retrieved_context_ratings": retrieved_context_ratings,
                "similarity_scores": similarity_scores,
        }


def process_context(context):
        formatted_context = ""
        for id, chunk, rating in zip(context['retrieved_context_ids'], context['retrieved_context'], context['retrieved_context_ratings']):
                formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
        return formatted_context


def build_pompt(preprocessed_context, question):
        prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.
You will be given a question and a list of context

Instrctions:
- You need to answer the question based on the provided context only
- Never use word context and refer to it as the available products

Context:
{preprocessed_context}

Question:
{question}
        """
        return prompt


def generate_answer(prompt):
        message, raw_response = instructor_client.messages.create_with_completion(
                max_tokens=2000,
                messages=[
                        {
                        "role": "user",
                        "content": prompt,
                        }
                ],
                model="claude-haiku-4-5",
                temperature=0,
                response_model=RAGGenerationResponse,
        )
        return message


def rag_pipeline(question, qdrant_client, top_k=5):
        load_dotenv()
        VOYAGE_API_KEY = os.environ.get("VOYAGE_API_KEY")
        voyageai_client = voyageai.Client(api_key=VOYAGE_API_KEY)
        retrieved_context = retrieve_data(voyageai_client, question, qdrant_client, top_k)
        preprocessed_context = process_context(retrieved_context)
        prompt = build_pompt(preprocessed_context, question)
        answer = generate_answer(prompt)

        # for evaluation we should return the following
        final_result = {
                "datamodel": answer,
                "answer": answer.answer,
                "question": question,
                "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
                "retrieved_context": retrieved_context["retrieved_context"],
                "similarity_scores": retrieved_context["similarity_scores"]
        }

        return final_result

In [13]:
question = "Can I get a charging cable? Please suggest me a good one."
output = rag_pipeline(question, qdrant_client)

In [14]:
output

{'datamodel': RAGGenerationResponse(answer="Yes, we have excellent charging cable options available! Here are my recommendations:\n\n**Best Overall Charger Solution:**\n**iPhone Fast Charger (ID: B0C6KVXZH9)** - Rating: 4.5/5\nThis is our top-rated option. It includes a 20W PD Type C Power Wall Charger with a 3FT Lightning Cable. It's Apple MFi Certified and offers 4X faster charging than regular chargers. It's compatible with iPhone 13/13 Pro Max/12/12 Mini/12 Pro/12 Pro Max/11/11 Pro, iPad, and AirPods. It also features built-in safety protection with real-time temperature and voltage monitoring.\n\n**Best Value - Cable Only:**\n**PEAPOLET iPhone Charger Cable (ID: B09NCXYHMV)** - Rating: 4.4/5\nThis is a 3-pack of 3.3FT MFi Certified USB-A to Lightning cables. Perfect if you already have a charger and just need quality cables. They support fast charging (2.4A) and fast data transfer (480Mbps), with a durable design that can withstand 10,000+ bends.\n\n**For USB-C Devices:**\n**Arae 

In [15]:
print(output['answer'])

Yes, we have excellent charging cable options available! Here are my recommendations:

**Best Overall Charger Solution:**
**iPhone Fast Charger (ID: B0C6KVXZH9)** - Rating: 4.5/5
This is our top-rated option. It includes a 20W PD Type C Power Wall Charger with a 3FT Lightning Cable. It's Apple MFi Certified and offers 4X faster charging than regular chargers. It's compatible with iPhone 13/13 Pro Max/12/12 Mini/12 Pro/12 Pro Max/11/11 Pro, iPad, and AirPods. It also features built-in safety protection with real-time temperature and voltage monitoring.

**Best Value - Cable Only:**
**PEAPOLET iPhone Charger Cable (ID: B09NCXYHMV)** - Rating: 4.4/5
This is a 3-pack of 3.3FT MFi Certified USB-A to Lightning cables. Perfect if you already have a charger and just need quality cables. They support fast charging (2.4A) and fast data transfer (480Mbps), with a durable design that can withstand 10,000+ bends.

**For USB-C Devices:**
**Arae USB Type C to 3.5mm Headphone and Charger Adapter (ID: 

### RAG Pipeline with Grounding Context

In [4]:
class RAGUsedContext(BaseModel):
        id: str = Field(description="The ID of the item used to answer the question")
        description: str = Field(description="Short description of the item used to answe the question")

class RAGGenerationResponse(BaseModel):
        answer: str = Field(description="The answer to the question")
        references: list[RAGUsedContext] = Field(description="List of items used to answer the question")

In [ ]:
instructor_client = instructor.from_anthropic(anthropic.Anthropic())
anthropic_client = anthropic.Anthropic()
qdrant_client = QdrantClient(url='http://localhost:6333')
def get_embedding(voyageai_client, text, model = 'voyage-3'):
        result = voyageai_client.embed(
                [text],
                model=model,
                input_type="document"
        )
        return result.embeddings[0]


def retrieve_data(voyageai_client, query, qdrant_client, k=5):
        query_embedding = get_embedding(voyageai_client, query)
        results = qdrant_client.query_points(
                collection_name="Amazon-items-collection-00",
                query=query_embedding,
                limit=k,
        )

        retrieved_context_ids = []
        retrieved_context = []
        similarity_scores = []
        retrieved_context_ratings = []

        for item in results.points:
                retrieved_context_ids.append(item.payload['parent_asin'])
                retrieved_context.append(item.payload['description'])
                retrieved_context_ratings.append(item.payload['average_rating'])
                similarity_scores.append(item.score)
        
        return {
                "retrieved_context_ids": retrieved_context_ids,
                "retrieved_context": retrieved_context,
                "retrieved_context_ratings": retrieved_context_ratings,
                "similarity_scores": similarity_scores,
        }


def process_context(context):
        formatted_context = ""
        for id, chunk, rating in zip(context['retrieved_context_ids'], context['retrieved_context'], context['retrieved_context_ratings']):
                formatted_context += f"- ID: {id}, rating: {rating}, description: {chunk}\n"
        return formatted_context


def build_pompt(preprocessed_context, question):
        prompt = f"""
You are a shopping assistant that can answer questions about the products in stock.
You will be given a question and a list of context

Instrctions:
- You need to answer the question based on the provided context only
- Never use word context and refer to it as the available products
- As an output you need to provide:
        * The answer to the question based on the provided context.
        * The list of the IDs of the chunks that were used to answer the question. Only return the ones that are used in the answer
        * Short description (1-2 sentences) of the item based on the description provided in the context 
- The answer description should have name of the item
- The answer to the question should contain detailed information about the product and returned with the detailed specification in bullet points

Context:
{preprocessed_context}

Question:
{question}
        """
        return prompt


def generate_answer(prompt):
        message, raw_response = instructor_client.messages.create_with_completion(
                max_tokens=2000,
                messages=[
                        {
                        "role": "user",
                        "content": prompt,
                        }
                ],
                model="claude-haiku-4-5",
                temperature=0,
                response_model=RAGGenerationResponse,
        )
        return message


def rag_pipeline(question, qdrant_client, top_k=5):
        load_dotenv()
        VOYAGE_API_KEY = os.environ.get("VOYAGE_API_KEY")
        voyageai_client = voyageai.Client(api_key=VOYAGE_API_KEY)
        retrieved_context = retrieve_data(voyageai_client, question, qdrant_client, top_k)
        preprocessed_context = process_context(retrieved_context)
        prompt = build_pompt(preprocessed_context, question)
        answer = generate_answer(prompt)

        # for evaluation we should return the following
        final_result = {
                "original_output": answer,
                "answer": answer.answer,
                "references": answer.references,
                "question": question,
                "retrieved_context_ids": retrieved_context["retrieved_context_ids"],
                "retrieved_context": retrieved_context["retrieved_context"],
                "similarity_scores": retrieved_context["similarity_scores"]
        }

        return final_result

In [16]:
question = "Can I get a charging cable? Please suggest me a good one."
output = rag_pipeline(question, qdrant_client)

In [19]:
output

{'original_output': RAGGenerationResponse(answer='Based on the available products, I would recommend the **iPhone Fast Charger by Hcoob** which has the highest rating (4.5 stars) among the charging options. Here are the details:\n\n**iPhone Fast Charger [Apple MFi Certified] 20W PD Type C Power Wall Charger with 3FT Lightning Cable**\n\n• **Apple MFi Certified** - Ensures compatibility and safety with Apple devices\n• **20W Fast Charging** - PD 3.0 quick charge technology, 4X faster than regular chargers\n• **3FT Lightning Cable Included** - Type C to Lightning cable with charging speed up to 3A\n• **High-Speed Data Transfer** - Up to 480Mb/s transfer speed\n• **Wide Compatibility** - Works with iPhone 13/12/11/X/8 series, iPad, and AirPods\n• **Built-in Safety Protection** - Monitors temperature and voltage in real time\n• **Smart Chip Technology** - Actively adjusts power output according to device needs\n• **Lifetime Service** - Manufacturer offers lifetime customer support\n\nAnoth

In [18]:
print(output['answer'])

Based on the available products, I would recommend the **iPhone Fast Charger by Hcoob** which has the highest rating (4.5 stars) among the charging options. Here are the details:

**iPhone Fast Charger [Apple MFi Certified] 20W PD Type C Power Wall Charger with 3FT Lightning Cable**

• **Apple MFi Certified** - Ensures compatibility and safety with Apple devices
• **20W Fast Charging** - PD 3.0 quick charge technology, 4X faster than regular chargers
• **3FT Lightning Cable Included** - Type C to Lightning cable with charging speed up to 3A
• **High-Speed Data Transfer** - Up to 480Mb/s transfer speed
• **Wide Compatibility** - Works with iPhone 13/12/11/X/8 series, iPad, and AirPods
• **Built-in Safety Protection** - Monitors temperature and voltage in real time
• **Smart Chip Technology** - Actively adjusts power output according to device needs
• **Lifetime Service** - Manufacturer offers lifetime customer support

Another good option is the **PEAPOLET iPhone Charger Cable** (4.4 st